# Cellpose-SAM: 920-nm green-channel session masks

This notebook follows the multi-mouse project layout:

`molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/920/preprocessing/green.tif`

and writes the labeled Cellpose mask to the canonical molecular-tracking location:

`molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/920/segmentation/mask.tif`

Only the 920-nm green channel is segmented. Existing `mask.tif` files are skipped by default.
The saved TIFF is a labeled `ZYX` volume, matching the format expected by the molecular-tracking repo.


In [14]:
from pathlib import Path
import csv
import os
import uuid

import numpy as np
import tifffile
from tqdm.auto import tqdm
from cellpose import models
from cellpose.io import imread_3D

In [15]:
# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
DERIVATIVES_ROOT = Path(r"D:\_data\_newAAV_2026\molecular_tracking_derivatives")

# None = process every mouse that has session-based 920 green images.
# Or restrict to selected mice, for example:
MOUSE_IDS = ["Fucci-Dead_2", "Fucci-Dead_1", "Fucci-Tri_3"]
# MOUSE_IDS = None

OVERWRITE_EXISTING = False
MIN_SIZE = 100

if not DERIVATIVES_ROOT.is_dir():
    raise FileNotFoundError(f"Derivatives root was not found: {DERIVATIVES_ROOT}")


In [16]:
# Discover only:
# <mouse>/sessions/<YYYYMMDD>/920/preprocessing/green.tif
green_paths = []

mouse_dirs = sorted(
    p for p in DERIVATIVES_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith("_")
)

if MOUSE_IDS is not None:
    wanted = set(MOUSE_IDS)
    mouse_dirs = [p for p in mouse_dirs if p.name in wanted]

for mouse_dir in mouse_dirs:
    sessions_dir = mouse_dir / "sessions"
    if not sessions_dir.is_dir():
        continue

    for session_dir in sorted(p for p in sessions_dir.iterdir() if p.is_dir()):
        green_path = session_dir / "920" / "preprocessing" / "green.tif"
        if green_path.is_file():
            green_paths.append(green_path)

if not green_paths:
    raise FileNotFoundError(
        "No session-based 920 green.tif files were found beneath:\n"
        f"{DERIVATIVES_ROOT}"
    )

print(f"Found {len(green_paths)} 920-nm green stack(s):")
for path in green_paths:
    mouse_id = path.parents[4].name
    session_id = path.parents[2].name
    print(f"  {mouse_id} / {session_id} -> {path}")


Found 23 920-nm green stack(s):
  Fucci-Dead_1 / 20260819 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260819\920\preprocessing\green.tif
  Fucci-Dead_1 / 20260820 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260820\920\preprocessing\green.tif
  Fucci-Dead_1 / 20260821 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260821\920\preprocessing\green.tif
  Fucci-Dead_1 / 20260824 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260824\920\preprocessing\green.tif
  Fucci-Dead_1 / 20260825 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260825\920\preprocessing\green.tif
  Fucci-Dead_1 / 20260827 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260827\920\preprocessing\green.tif
  Fucci-Dead_1 / 20260828 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260828\920\prepr

In [17]:
# Same Cellpose-SAM model/settings as the working 1050-nm red-channel notebook.
model = models.CellposeModel(
    gpu=True,
    pretrained_model="cpsam_v2",
)


In [18]:
results = []

for green_path in tqdm(green_paths, desc="Cellpose-SAM 920 green sessions"):
    mouse_id = green_path.parents[4].name
    session_id = green_path.parents[2].name

    segmentation_dir = green_path.parent.parent / "segmentation"
    output_path = segmentation_dir / "mask.tif"

    if output_path.exists() and not OVERWRITE_EXISTING:
        tqdm.write(f"Skipping existing: {mouse_id} {session_id} -> {output_path}")
        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "green_path": str(green_path),
            "mask_path": str(output_path),
            "status": "ALREADY_EXISTS",
            "n_masks": "",
            "error": "",
        })
        continue

    temp_path = None

    try:
        tqdm.write(f"Processing: {mouse_id} {session_id}")

        loaded_image = imread_3D(green_path.as_posix())

        masks, flows, styles = model.eval(
            loaded_image,
            do_3D=True,
            z_axis=0,
            channel_axis=3,
            min_size=MIN_SIZE,
        )

        n_masks = int(np.max(masks))
        tqdm.write(f"  Found {n_masks} mask(s)")

        segmentation_dir.mkdir(parents=True, exist_ok=True)

        # Canonical molecular-tracking mask: integer-labeled ZYX volume.
        # uint16 is sufficient unless there are >65,535 labels.
        mask_dtype = np.uint16 if n_masks <= np.iinfo(np.uint16).max else np.uint32
        mask_to_save = masks.astype(mask_dtype, copy=False)

        temp_path = segmentation_dir / f".mask.tif.tmp.{uuid.uuid4().hex}"

        tifffile.imwrite(
            temp_path,
            mask_to_save,
            photometric="minisblack",
            metadata={"axes": "ZYX"},
        )

        # Read the temporary file back before promotion.
        check = tifffile.imread(temp_path)
        if check.shape != mask_to_save.shape:
            raise RuntimeError(
                f"Saved mask shape mismatch: expected {mask_to_save.shape}, got {check.shape}"
            )

        if output_path.exists():
            raise FileExistsError(
                f"Destination appeared during processing; refusing to overwrite: {output_path}"
            )

        # On Windows os.rename refuses to replace an existing destination.
        os.rename(temp_path, output_path)
        temp_path = None

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "green_path": str(green_path),
            "mask_path": str(output_path),
            "status": "SEGMENTED",
            "n_masks": n_masks,
            "error": "",
        })

        tqdm.write(f"Saved: {output_path}")

    except Exception as error:
        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "green_path": str(green_path),
            "mask_path": str(output_path),
            "status": "FAILED",
            "n_masks": "",
            "error": str(error),
        })

        tqdm.write(f"FAILED: {mouse_id} {session_id}")
        tqdm.write(f"  {error}")


Cellpose-SAM 920 green sessions:   0%|          | 0/23 [00:00<?, ?it/s]

Skipping existing: Fucci-Dead_1 20260819 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260819\920\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260820 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260820\920\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260821 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260821\920\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260824 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260824\920\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260825 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260825\920\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260827 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260827\920\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260828 -> D:\_data\_newAAV_2026\molecular_

Cellpose-SAM 920 green sessions: 100%|██████████| 23/23 [14:18<00:00, 37.32s/it]

  Found 2009 mask(s)
Saved: D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Tri_3\sessions\20260904\920\segmentation\mask.tif


In [19]:
log_path = DERIVATIVES_ROOT / "cellposeSAM_920_green_session_log.csv"

with log_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "mouse_id",
            "session_id",
            "green_path",
            "mask_path",
            "status",
            "n_masks",
            "error",
        ],
    )
    writer.writeheader()
    writer.writerows(results)

print("\nFinished.")
print(f"Log: {log_path}")

status_counts = {}
for row in results:
    status_counts[row["status"]] = status_counts.get(row["status"], 0) + 1

for status, count in sorted(status_counts.items()):
    print(f"{status:20s} {count:4d}")

failed = [row for row in results if row["status"] == "FAILED"]
if failed:
    print("\nFailed sessions:")
    for row in failed:
        print(f"  {row['mouse_id']} {row['session_id']}: {row['error']}")



Finished.
Log: D:\_data\_newAAV_2026\molecular_tracking_derivatives\cellposeSAM_920_green_session_log.csv
ALREADY_EXISTS         22
SEGMENTED               1
